# Étude de cas : la troisième loi de Kepler, avec les unités

Vous donnez à GP_ELITE le demi-grand axe et la période des huit planètes, en
mètres et en secondes, et vous lui dites ces deux unités, rien d'autre. Il rend
la loi T = c·a^1,5 et les unités de c, qu'il déduit seul : s/m^1,5. De c se
déduit le produit de la constante de gravitation et de la masse du Soleil,
G·M☉ = (2π/c)². Ajustée sur les quatre planètes intérieures seulement, la loi
donne G·M☉ à 0,002 % près de la valeur de l'Union astronomique internationale
(UAI), et les périodes des quatre planètes extérieures à 0,28 % près.

Ce notebook dit d'où viennent les données, ce que le moteur rend, ce qu'il ne
trouve pas, et vérifie la loi sur des planètes tenues à l'écart de
l'ajustement. Chaque chiffre du texte a été mesuré avec
`benchmarks/case_studies.py kepler`, pour les graines 0 à 4 (sortie dans
`benchmarks/results_0.9/case_studies/kepler.txt`, et avec la version 1.1.0 dans
`benchmarks/results_1.1/release/m3/case_kepler.txt`).

Exécutez les cellules dans l'ordre avec **Maj + Entrée**. Chaque ajustement
prend de moins d'une seconde à une dizaine de secondes.

*[🇬🇧 English version](https://colab.research.google.com/github/ariel95500-create/gp-elite/blob/main/examples/case_studies/kepler_units.ipynb)*

## 1. Installation

In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'prêt.')

## 2. Les données et leur source

Le tableau n'est pas recopié d'un manuel. C'est la table des éléments orbitaux
moyens de Simon et de ses collègues (1994, *Astronomy and Astrophysics* 282,
663), telle que la distribue la bibliothèque d'astronomie ERFA (fichier
`plan94.c` du paquet source pyerfa 2.0.1.5 sur PyPI). Pour chaque planète elle
donne le demi-grand axe *a*, en unités astronomiques, et le moyen mouvement, en
secondes d'arc par millénaire. La période s'en déduit : un tour complet,
1 296 000 secondes d'arc, divisé par les secondes d'arc parcourues en un an. La
ligne de la Terre est le centre de masse de la Terre et de la Lune.

In [ ]:
import os
import numpy as np
import pandas as pd

NAME = 'planets_simon1994.csv'
URL = ('https://raw.githubusercontent.com/ariel95500-create/gp-elite/main/'
       'examples/case_studies/data/' + NAME)
path = os.path.join('data', NAME)
df = pd.read_csv(path if os.path.exists(path) else URL, comment='#')

# 1 296 000 secondes d'arc dans un tour, 1000 ans dans un millénaire
df['period_yr'] = 1296000000.0 / df['mean_motion']
df['a3_over_T2'] = df['a_au']**3 / df['period_yr']**2
print(df[['planet', 'a_au', 'period_yr', 'a3_over_T2']].to_string(index=False))

**À lire avant les résultats.** Ce ne sont pas des observations brutes. Les
éléments moyens sortent d'une théorie du mouvement des planètes ajustée sur des
siècles d'observations, et la dernière colonne montre que *a* et *T* suivent
déjà la loi de Kepler : a³/T² vaut de 0,99994 à 0,99997 de Mercure à Mars,
1,0008 pour Jupiter, et de 1,0053 à 1,0055 de Saturne à Neptune. Cette étude ne
vérifie donc pas que Kepler avait raison. Elle vérifie ce que le moteur trouve
dans un tel tableau, et ce qu'il déduit des unités.

## 3. Avec les unités : la loi et sa constante

On dit au moteur que *a* est en mètres et *T* en secondes
(`units=['m'], target_units='s'`), et que la loi contient une constante dont il
doit trouver les unités (`unknown_constant=True`). La conversion utilise l'unité
astronomique, 149 597 870 700 m exactement (UAI 2012), et l'année julienne,
365,25 jours.

In [ ]:
from gp_elite import GPEliteRegressor

AU = 149597870700.0          # mètres dans une unité astronomique (UAI 2012)
YEAR = 365.25 * 86400.0      # secondes dans une année julienne
GM_IAU = 1.3271244e20        # m^3/s^2, valeur nominale (UAI 2015)

a = df['a_au'].to_numpy() * AU
T = df['period_yr'].to_numpy() * YEAR

est = GPEliteRegressor(units=['m'], target_units='s', unknown_constant=True,
                       generations=30, speed='fast', random_state=0,
                       parallel=False)
est.fit(a.reshape(-1, 1), T)

c = est.constant_value_
GM = (2 * np.pi / c) ** 2
print('loi       :', est.equation_)
print('constante :', c, est.constant_units_string())
print(f'G * M_sol : {GM:.6e} m^3/s^2   ({100 * (GM / GM_IAU - 1):+.3f} % de la valeur UAI)')

Vous devriez lire `5.43909e-10 * X0 * sqrt(X0)`, c'est-à-dire c·a^1,5, avec c en
s/m^1,5 : le moteur a trouvé les unités de la constante à partir de celles des
données. Les graines 0 à 4 rendent toutes cette formule.

Avec les unités, un nombre ajouté à c·a^1,5 serait ajouté à une durée sans en
être une : le moteur ne peut pas l'écrire (comparez avec la section 5).

De c, G·M☉ = 1,33447e20 m³/s², 0,55 % au-dessus de la valeur de l'UAI,
1,3271244e20. L'écart vient du tableau, pas de la recherche : ajustée sur les
huit lignes, la loi est tirée par les longues périodes de Saturne, d'Uranus et
de Neptune, dont le a³/T² est 0,5 % au-dessus de celui des planètes intérieures
(section 2).

## 4. Une vérification tenue à l'écart : les planètes intérieures prédisent les extérieures

Le même estimateur, ajusté sur Mercure, Vénus, la Terre et Mars seulement.
Jupiter, Saturne, Uranus et Neptune ne sont jamais vues : leurs périodes sont
prédites à partir de leurs distances.

In [ ]:
inner, outer = slice(0, 4), slice(4, 8)

est_in = GPEliteRegressor(units=['m'], target_units='s', unknown_constant=True,
                          generations=30, speed='fast', random_state=0,
                          parallel=False)
est_in.fit(a[inner].reshape(-1, 1), T[inner])

c_in = est_in.constant_value_
GM_in = (2 * np.pi / c_in) ** 2
print('loi, de Mercure à Mars :', est_in.equation_)
print('constante :', c_in, est_in.constant_units_string())
print(f'G * M_sol : {GM_in:.6e} m^3/s^2   ({100 * (GM_in / GM_IAU - 1):+.4f} % de la valeur UAI)')
print()
pred = est_in.predict(a[outer].reshape(-1, 1))
for name, p, t in zip(df['planet'][outer], pred, T[outer]):
    print(f'{name:8s} prédite {p / YEAR:9.4f} ans   table {t / YEAR:9.4f} ans   '
          f'écart {100 * (p / t - 1):+.3f} %')

Les cinq graines rendent la même loi, `5.45416e-10 * X0 * sqrt(X0)`, avec
c = 5,45416e-10 s/m^1,5. La version 1.0 rendait la même constante, et la même
formule pour la graine 3, mais écrivait a^1,5 autrement pour les quatre autres
graines : comme un produit de quatre racines de *a*, comme `sqrt(X0) * X0` ou
`sqrt(X0 * X0 * X0)`, ou comme la différence de deux termes en a^1,5. La
constante que le moteur annonce est le coefficient de la loi
telle qu'elle prédit, quelle que soit l'écriture.

G·M☉ = 1,327103e20 m³/s², 0,0016 % sous la valeur de l'UAI. Les périodes des
planètes extérieures sortent trop longues de 0,043 % pour Jupiter, et de 0,27 %
à 0,28 % pour Saturne, Uranus et Neptune : l'écart que leur a³/T² montre déjà
dans le tableau.

## 5. Sans les unités

Le même tableau, en unités astronomiques et en années, sans rien dire au moteur
des unités.

In [ ]:
x_au = df[['a_au']].to_numpy()
P = df['period_yr'].to_numpy()

plain = GPEliteRegressor(generations=30, speed='fast', random_state=0,
                         parallel=False)
plain.fit(x_au, P)
print('loi sans unités :', plain.equation_)
print()
for name, p, t in zip(df['planet'], plain.predict(x_au), P):
    print(f'{name:8s} écart {100 * (p / t - 1):+.3f} %')

Sans les unités, le moteur rend `0.00684983 + 0.997207 * X0 * sqrt(X0)` : la même
puissance, plus 0,0068 an (2,5 jours) que rien en physique ne justifie. La
recherche minimise l'erreur au carré sur les périodes, que dominent les longues
périodes : cette formule ramène Saturne, Uranus et Neptune à moins de 0,01 %, et
coûte 2,6 % sur la période de Mercure et 0,8 % sur celle de Vénus. Les cinq
graines la rendent. Avec les unités, ce terme ne peut pas s'écrire.

Ajusté sur les quatre planètes intérieures seulement, sans les unités, le moteur
rend `-7.16378e-06 + 1.00003 * X0 * sqrt(X0)`, et manque les périodes
extérieures des mêmes 0,04 % et 0,27 % à 0,28 % qu'avec les unités.

## 6. Ce que le moteur ne trouve pas

- **Pourquoi a³/T² change d'une planète à l'autre.** Dans ce tableau, il varie
  jusqu'à 0,56 %. Pour Jupiter, dont la masse est un millième de celle du
  Soleil (1/1047,355, dans la
  table des masses du même fichier source, recopiée en tête du fichier de
  données),
  la forme de Newton de la loi, a³/T²
  proportionnel à M☉ + m, attend environ 0,1 % ; le tableau montre 0,08 %. Pour
  Saturne, Uranus et Neptune, dont les masses sont bien plus petites, les 0,5 %
  sont dans le tableau et le moteur ne peut pas dire d'où ils viennent : une loi
  de *a* seul, avec une seule constante, ne peut pas les suivre.
- **G et la masse du Soleil séparément.** Les données ne contiennent que leur
  produit.
- **Le 2π.** Le moteur rend c ; que c = 2π/√(G·M☉), c'est la physique que nous
  apportons.

## 7. Les cinq graines

Les graines 0 à 4 rendent les mêmes constantes, au dernier chiffre affiché près,
dans chaque ajustement de ce notebook, et depuis la version 1.1 les mêmes
formules. Les chiffres ont été mesurés sur la
branche de développement qui suit la version 0.9.0, puis de nouveau sur le
paquet des versions 1.0.0 et 1.1.0, avec les mêmes constantes au dernier chiffre
affiché près (`benchmarks/results_0.9/release_1.0/m3/`,
`benchmarks/results_1.1/release/m3/`).

La sortie complète : `benchmarks/results_1.1/release/m3/case_kepler.txt`
(version 1.1.0), et `benchmarks/results_0.9/case_studies/kepler.txt` pour la
première mesure. Le
protocole, écrit avant les mesures : `benchmarks/results_0.9/PLAN_PHASE6.md`.